# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. Analizar si el negocio es rentable (revenue, costos y profit)  

3. Entender dónde se pierden los usuarios (funnel de conversión)  

4. Evaluar si los usuarios regresan (retención por cohortes)  

5. Validar si los cambios generan impacto (test estadístico)  

6. Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv') 
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
print(orders.shape)
orders.info()
print(orders.columns)

(25100, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
Index(['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo',
       'fuente_referencia', 'nombre_producto', 'categoria_produ

In [4]:
# Se corrigió el tipo de dato de fecha_hora_pedido de object a datetime usando errors = ‘coerce’, para marcar como NaT cualquier valor que no se pueda convertir.

orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
print(orders['fecha_hora_pedido'].dtype)
print(orders['fecha_hora_pedido'].isna().sum())

datetime64[ns]
0


In [5]:
# Revisar duplicados en id_pedido (debe ser único por pedido)
print(orders['id_pedido'].duplicated().sum())

# Revisar filas completamente idénticas en todas las columnas
print(orders.duplicated().sum())

# Ver todas las filas involucradas en duplicados de id_pedido (no solo la repetida)
orders[orders['id_pedido'].duplicated(keep=False)]

100
100


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
734,order_734,user_6347,2025-05-02,Colombia,desktop,organic,Blender-XL-Red,Hogar,1.0,58.00,0.0,58.00
812,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32
974,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16
1361,order_1361,user_507,2025-06-27,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,239.61,0.0,479.22
1735,order_1735,user_728,2025-05-13,Mexico,mobile,social,Tablet-Standard-64GB,Electronica,2.0,419.04,0.0,838.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25095,order_3913,user_380,2025-02-18,Argentina,desktop,paid_search,Phone-Pro-128GB,Electronica,1.0,82.28,0.0,82.28
25096,order_23405,user_7833,2025-04-04,Colombia,mobile,paid_search,Phone-Pro-128GB,Electronica,2.0,99.25,5.0,193.50
25097,order_5615,user_5417,2025-05-13,Colombia,desktop,social,Blender-XL-Red,Hogar,2.0,450.35,5.0,895.69
25098,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32


In [6]:
# Verificar si los nulos de cantidad, precio_unitario y monto_descuento coinciden en las mismas filas 
# Se detectaron 50 filas con valores nulos simultáneos en cantidad, precio_unitario y monto_descuento (confirmado con .isna() combinado)
# Posible problema sistemático de captura, ya que coinciden en las mismas filas
# Verificar si los nulos de cantidad, precio_unitario, y monto_descuento, coinciden en las mismas filas

orders[orders['cantidad'].isna() & orders['precio_unitario'].isna() & orders['monto_descuento'].isna()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,527.15
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,711.18
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,NaN,NaN,NaN,NaN,41.08
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,431.55
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,279.35


In [7]:
# Eliminar filas duplicadas, conservando la primera aparición de cada pedido
orders = orders.drop_duplicates(keep='first')
print(orders.shape)

(25000, 12)


In [8]:
# La columna ‘pais’ tenía inconsistencias de mayúsculas/minúsculas (ej. ‘Mexico vs ‘mexico’), esto duplicaba categorías
# Se normalizó con .str.title()

orders['pais'] = orders['pais'].str.title() 
orders['pais'].value_counts()

Mexico       8341
Colombia     8304
Argentina    8055
Name: pais, dtype: int64

In [9]:
# Se revisaron valores negativos o cero inválidos en variables numéricas 
print("Cantidad <= 0:", (orders['cantidad'] <= 0).sum())
print("Precio unitario <= 0:", (orders['precio_unitario'] <= 0).sum())
print("Monto descuento < 0:", (orders['monto_descuento'] < 0).sum())
print("Monto total <= 0:", (orders['monto_total'] <= 0).sum())

Cantidad <= 0: 4
Precio unitario <= 0: 0
Monto descuento < 0: 0
Monto total <= 0: 4


In [10]:
# Se detectaron 10 pedidos con cantidad anormalmente alta ( 10,000 y 20,000 unidades exactas)
# Se eliminaron porque representaban un error evidente de captura. Las 50 filas con cantidad nula
# (ya documentadas previamente) se conservan intactas para investigar posteriormente 

orders = orders[(orders['cantidad'] <= 100) | (orders['cantidad'].isna())]
print(orders.shape)

(24990, 12)


In [11]:
# Verificar que se eliminó con el filtro de cantidad <= 100
print((orders['cantidad'] > 100).sum())
print(orders['cantidad'].isna().sum())

0
50


In [12]:
# Confirmar si las filas con cantidad <= 0 son las mismas que monto_total <= 0
orders[(orders['cantidad'] <= 0) & (orders['monto_total'] <= 0)]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


In [13]:
# Se detectaron 4 filas con cantidad y monto_total negativos o cero (posible error de captura o registro de devolución mal clasificado). 

# Se eliminan por no poder confirmar el valor correcto y representar un impacto mínimo (4 de 25,000 filas)

orders = orders[~((orders['cantidad'] <= 0) & (orders['monto_total'] <= 0))]
print(orders.shape)

(24986, 12)


In [14]:
# Verificar consistencia entre monto_total y la formula esperada
orders['monto_esperado'] = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']
inconsistentes = orders[abs(orders['monto_total'] - orders['monto_esperado']) > 0.01]
print("Filas con monto_total inconsistente:", inconsistentes.shape[0])

Filas con monto_total inconsistente: 1142


In [15]:
# Ver el tamaño de las diferencias en los montos inconsistentes
inconsistentes = orders[abs(orders['monto_total'] - orders['monto_esperado']) > 0.01].copy()
inconsistentes['diferencia'] = orders['monto_total'] - orders['monto_esperado']
print(inconsistentes['diferencia'].describe())

count    1142.000000
mean        0.000420
std         0.009996
min        -0.010000
25%        -0.010000
50%         0.010000
75%         0.010000
max         0.010000
Name: diferencia, dtype: float64


In [16]:
# Se detectaron 1,142 filas donde monto_total no coincide exactamente con la fórmula.
# (cantidad * precio_unitario – monto_descuento. Las diferencias son mínimas (entre -0.01 y 0.01, promedio de 0.0004) – consistentes con redondeo de punto flotante, 
# no un error de calculo real.
# No se corrigen ni eliminan, solo se documenta el hallazgo.

print(inconsistentes.shape[0])

1142


In [17]:
# Revisar valores únicos en variables categóricas restantes
print("dispositivo:", orders['dispositivo'].unique())
print("fuente_referencia:", orders['fuente_referencia'].unique())
print("categoria_producto (orders):", orders['categoria_producto'].unique())
print("categoria_producto (catalog):", catalog['categoria_producto'].unique())
print("proveedor:", catalog['proveedor'].unique())

dispositivo: ['desktop' 'mobile' nan]
fuente_referencia: ['organic' 'paid_search' 'social' nan]
categoria_producto (orders): ['Moda' 'Electronica' 'Hogar' nan]
categoria_producto (catalog): ['Electrónica' 'Hogar' 'Moda']
proveedor: ['Fuller, Pena and Myers' 'King Ltd' 'Bowers LLC' 'Long-Reid'
 'Rivera, Carr and Finley' 'Greene-Smith' 'Mcmillan-Rhodes']


In [18]:
# Estandarizar ‘categoria_priducto’ sin acentos para que coincida entre orders y catalog
catalog['categoria_producto'] = catalog['categoria_producto'].str.replace('Electrónica', 'Electronica')

# Se verifica que coincidan
print(sorted(orders['categoria_producto'].dropna().unique()))
print(sorted(catalog['categoria_producto'].unique()))

['Electronica', 'Hogar', 'Moda']
['Electronica', 'Hogar', 'Moda']


In [19]:
print(catalog.shape)
catalog.info()
print(catalog.columns)

(7, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
Index(['nombre_producto', 'categoria_producto', 'costo_unitario', 'proveedor'], dtype='object')


In [20]:
# Se omparan productos únicos de orders y catalog para confirmar que se pueden cruzar sin problema
set(orders['nombre_producto'].unique()) - set(catalog['nombre_producto'].unique())

{nan}

In [21]:
# Se aíslan filas con nombre_producto nulo.
#Se detectaron 30 filas con nombre_producto y categoría_producto nulos, pero con cantidad, precio_unitario, monto_descuento y monto_total completos. 
# Es un grupo distinto en filas problemáticas al de los 50 nulos en cantidad/precio/descuento (no se traslapan), 
# es un problema posible separado en la captura de datos del producto.

print(orders[orders['nombre_producto'].isna()].shape[0])
orders[orders['nombre_producto'].isna()]

30


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado
44,order_44,user_2899,2025-02-22,Colombia,desktop,NaN,NaN,NaN,1.0,318.38,5.0,313.38,313.38
45,order_45,user_6196,2025-02-10,Mexico,mobile,NaN,NaN,NaN,2.0,354.06,10.0,698.12,698.12
46,order_46,user_5815,2025-04-24,Mexico,mobile,NaN,NaN,NaN,1.0,359.31,10.0,349.31,349.31
47,order_47,user_406,2025-05-08,Colombia,mobile,NaN,NaN,NaN,2.0,476.78,0.0,953.56,953.56
48,order_48,user_6187,2025-01-26,Colombia,desktop,NaN,NaN,NaN,2.0,137.96,0.0,275.93,275.92
49,order_49,user_7225,2025-04-04,Argentina,desktop,NaN,NaN,NaN,2.0,433.76,0.0,867.53,867.52
50,order_50,user_1840,2025-02-05,Mexico,desktop,NaN,NaN,NaN,1.0,495.38,5.0,490.38,490.38
51,order_51,user_5998,2025-02-21,Mexico,desktop,NaN,NaN,NaN,2.0,382.94,0.0,765.89,765.88
52,order_52,user_5679,2025-04-17,Argentina,mobile,NaN,NaN,NaN,2.0,415.01,0.0,830.01,830.02
53,order_53,user_5936,2025-01-02,Argentina,desktop,NaN,NaN,NaN,2.0,236.49,0.0,472.98,472.98


In [22]:
# Se revisan duplicados en catalog
print(catalog['nombre_producto'].duplicated().sum())
print(catalog.duplicated().sum())

0
0


In [23]:
print("Costo unitario <= 0:", (catalog['costo_unitario'] <= 0).sum())

Costo unitario <= 0: 0


In [24]:
print(marketing.shape)
marketing.info()
print(marketing.columns)

(1620, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB
Index(['fecha', 'pais', 'id_campaña', 'canal', 'gasto'], dtype='object')


In [25]:
# Se corrigió el tipo de dato de fecha en marketing, de object a datetime
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
print(marketing['fecha'].dtype)
print(marketing['fecha'].isna().sum())

datetime64[ns]
0


In [26]:
# Se detectaron 101 valores nulos en ‘canal’
# Se documenta el hallazgo y se decide más adelante si se excluyen del desglose por canal

print(marketing['canal'].isna().sum())

101


In [27]:
# Se revisan duplicados en marketing
print(marketing.duplicated().sum())

0


In [28]:
print("Gasto < 0:", (marketing['gasto'] < 0).sum())

Gasto < 0: 0


---

### Revisión y calidad de datos

**Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

In [29]:
# La limpieza de datos (fechas, variables numéricas, consistencia de montos, duplicados y variables categóricas) se realizó directamente en los bloques de orders,
# catalog y marketing más arriba en el notebook.

---
**Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [30]:
# Se exportan los datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

** Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

** Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

** Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [31]:
# El dataset no especifica la moneda de los montos. Se asume una moneda única, 
# ya que mezcla pedidos de Argentina, Colombia y México sin conversión cambiaria explícita. 

orders['monto_total'].sum()

9643909.559999999

In [32]:
# Traer el costo unitario a cada pedido
orders_con_costo = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')

# Calcular el costo total de cada pedido (costo unitario * cantidad)
orders_con_costo['costo_total_pedido'] = orders_con_costo['costo_unitario'] * orders_con_costo['cantidad']

# Sumar todos los pedidos para obtener el costo total del negocio
orders_con_costo['costo_total_pedido'].sum()

3828869.01

In [33]:
# Calcular profit y margen de rentabilidad (solo sobre pedidos con costo calculable)
pedidos_validos = orders_con_costo['costo_total_pedido'].notna()
profit = orders_con_costo.loc[pedidos_validos, 'monto_total'].sum() - orders_con_costo.loc[pedidos_validos, 'costo_total_pedido'].sum()
margen = (profit / orders_con_costo.loc[pedidos_validos, 'monto_total'].sum()) * 100

print('Profit:', profit)
print('Margen (%):', margen)

Profit: 5781149.93
Margen (%): 60.157528992341405


In [34]:
# Calcular el total invertido en marketing
gasto_total = marketing['gasto'].sum()
print("Gasto total en marketing:", gasto_total)

Gasto total en marketing: 2871843.53


In [35]:
# Profit neto considerando también el gasto en marketing
profit_neto = profit - gasto_total
margen_neto = (profit_neto / orders_con_costo.loc[pedidos_validos, 'monto_total'].sum()) * 100
print("Profit neto:", profit_neto)
print("Margen neto (%):", margen_neto)

Profit neto: 2909306.4
Margen neto (%): 30.27368018902156


In [36]:
orders['monto_total'].mean()

385.97252701512843

In [37]:
orders['cantidad'].mean()

1.5049326275264678

In [38]:
marketing.groupby('canal')['gasto'].sum()

canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64

In [39]:
orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)

nombre_producto
Vacuum-Pro-Black        6284.0
Blender-XL-Red          6279.0
Jacket-Winter-M         6256.0
Sneakers-Urban-42       6172.0
Laptop-Gaming-16GB      4198.0
Tablet-Standard-64GB    4153.0
Phone-Pro-128GB         4140.0
Name: cantidad, dtype: float64

In [40]:
# Investigar la distribución de cantidad para Laptop-Gaming-16GB
laptop = orders[orders['nombre_producto'] == 'Laptop-Gaming-16GB']
print(laptop['cantidad'].describe())

count    2768.000000
mean        1.516618
std         0.499814
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         2.000000
Name: cantidad, dtype: float64


In [41]:
# Aislar pedidos de Laptop-Gaming-16GB con cantidad anormalmente alta
laptop[laptop['cantidad'] > 100]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado


---

## Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

** Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

** Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

** Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [42]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [43]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [44]:
# PARTE 1: Totales del funnel
# ======================

# PARTE 1: Totales del funnel
query_totals = '''
SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
'''
totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,add_payment_info,6250
1,add_to_cart,7634
2,begin_checkout,7208
3,first_visit,7796
4,purchase,6240
5,select_item,7582


In [45]:
query_orden = '''
SELECT nombre_evento, MIN(timestamp_evento) AS primer_registro
FROM events
GROUP BY nombre_evento
ORDER BY primer_registro
'''
orden = pd.read_sql(query_orden, con=engine)
orden

,nombre_evento,primer_registro
0,add_payment_info,2025-01-01
1,first_visit,2025-01-01
2,begin_checkout,2025-01-01
3,add_to_cart,2025-01-01
4,select_item,2025-01-01
5,purchase,2025-01-01


In [46]:
# NOTA: add_to_cart (7,634) tiene más usuarios únicos que select_item (7,582), rompiendo el orden esperado del funnel. 
# Posibles causas: falla de tracking en el evento select_item, o rutas alternativas del usuario 
# (compra rápida, productos recomendados agregados directo al carrito sin pasar por una vista de selección individual).


In [47]:
# PARTE 2: Conversiones

# Ordenar manualmente según el flujo lógico del funnel
orden_funnel = ['first_visit', 'select_item', 'add_to_cart', 'begin_checkout', 'add_payment_info', 'purchase']
totals_ordenado = totals.set_index('nombre_evento').loc[orden_funnel].reset_index()

# Calcular tasa de conversión entre cada etapa
totals_ordenado['tasa_conversion'] = (totals_ordenado['usuarios_unicos'] / totals_ordenado['usuarios_unicos'].shift(1)) * 100
totals_ordenado

,nombre_evento,usuarios_unicos,tasa_conversion
0,first_visit,7796,NaN
1,select_item,7582,97.255003
2,add_to_cart,7634,100.685835
3,begin_checkout,7208,94.419701
4,add_payment_info,6250,86.709212
5,purchase,6240,99.840000


In [48]:

# Se identifica el mayor cuello de botella de funnel ocurre entre begin_checkout y add_payment_info (94.42% -> 86.71%), donde se pierde la mayor proporción de usuarios.
# Nota: add_to_cart muestra una tasa de conversión de 100.69% respecto a select_item, lo cual no es lógico. 
# Esto confirma la anomalía de tracking documentada anteriormente. 


In [49]:
tasa_final = (6240 / 7796) * 100
print("Tasa de conversión final:", tasa_final)

Tasa de conversión final: 80.04104669061057


---

##  Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

** Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [50]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [51]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [52]:
user_activity['semana'] = (user_activity['dias_despues_registro'] / 7).astype(int)
user_activity.head()

,id_usuario,fecha_actividad,dias_despues_registro,activo,semana
0,user_0,2025-02-05,7,0,1
1,user_0,2025-02-12,14,1,2
2,user_0,2025-02-19,21,1,3
3,user_0,2025-02-26,28,0,4
4,user_1,2025-01-14,7,0,1


In [53]:
users['fecha_registro'].dtype

dtype('O')

In [54]:
users['fecha_registro'] = pd.to_datetime(users['fecha_registro'], errors='coerce')
print(users['fecha_registro'].dtype)

datetime64[ns]


In [55]:
users['mes_registro'] = users['fecha_registro'].dt.to_period('M')
users.head(5)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan,mes_registro
0,user_0,2025-01-29,Mexico,mobile,free,2025-01
1,user_1,2025-01-07,Mexico,mobile,free,2025-01
2,user_2,2025-03-12,Argentina,mobile,free,2025-03
3,user_3,2025-03-04,Mexico,desktop,free,2025-03
4,user_4,2025-02-27,Argentina,desktop,free,2025-02


In [56]:
retencion = user_activity.merge(users[['id_usuario', 'mes_registro']], on='id_usuario', how='left')
retencion.head()

,id_usuario,fecha_actividad,dias_despues_registro,activo,semana,mes_registro
0,user_0,2025-02-05,7,0,1,2025-01
1,user_0,2025-02-12,14,1,2,2025-01
2,user_0,2025-02-19,21,1,3,2025-01
3,user_0,2025-02-26,28,0,4,2025-01
4,user_1,2025-01-14,7,0,1,2025-01


In [57]:
retencion_semanal = retencion.groupby(['mes_registro', 'semana'])['activo'].sum().reset_index()
retencion_semanal

,mes_registro,semana,activo
0,2025-01,1,697
1,2025-01,2,668
2,2025-01,3,656
3,2025-01,4,671
4,2025-02,1,611
5,2025-02,2,609
6,2025-02,3,635
7,2025-02,4,575
8,2025-03,1,677
9,2025-03,2,705


In [58]:
retencion_semanal['semana1_valor'] = retencion_semanal.groupby('mes_registro')['activo'].transform('first')
retencion_semanal.head(10)

,mes_registro,semana,activo,semana1_valor
0,2025-01,1,697,697
1,2025-01,2,668,697
2,2025-01,3,656,697
3,2025-01,4,671,697
4,2025-02,1,611,611
5,2025-02,2,609,611
6,2025-02,3,635,611
7,2025-02,4,575,611
8,2025-03,1,677,677
9,2025-03,2,705,677


In [59]:
retencion_semanal['porcentaje_retencion'] = (retencion_semanal['activo'] / retencion_semanal['semana1_valor']) * 100
retencion_semanal

,mes_registro,semana,activo,semana1_valor,porcentaje_retencion
0,2025-01,1,697,697,100.000000
1,2025-01,2,668,697,95.839311
2,2025-01,3,656,697,94.117647
3,2025-01,4,671,697,96.269727
4,2025-02,1,611,611,100.000000
5,2025-02,2,609,611,99.672668
6,2025-02,3,635,611,103.927987
7,2025-02,4,575,611,94.108020
8,2025-03,1,677,677,100.000000
9,2025-03,2,705,677,104.135894


In [60]:
# Verificar si hay filas duplicadas en user_activity
print(user_activity.duplicated().sum())

0


In [61]:
# Filtrar la cohorte de marzo, semana 2, donde vimos retención > 100%
ejemplo = user_activity.merge(users[['id_usuario', 'mes_registro']], on='id_usuario', how='left')
ejemplo_filtrado = ejemplo[(ejemplo['mes_registro'] == '2025-03') & (ejemplo['semana'] == 2) & (ejemplo['activo'] == 1)]

print("Filas:", ejemplo_filtrado.shape[0])
print("Usuarios únicos:", ejemplo_filtrado['id_usuario'].nunique())

Filas: 705
Usuarios únicos: 705


In [62]:
# Tamaño real de cada cohorte (total de usuarios registrados por mes)
tamano_cohorte = users.groupby('mes_registro')['id_usuario'].nunique().reset_index()
tamano_cohorte.columns = ['mes_registro', 'total_usuarios_cohorte']
tamano_cohorte

,mes_registro,total_usuarios_cohorte
0,2025-01,1627
1,2025-02,1444
2,2025-03,1636
3,2025-04,1606
4,2025-05,1687


In [63]:
# Unir el tamaño real de cada cohorte a la tabla de retención semanal
retencion_semanal = retencion_semanal.merge(tamano_cohorte, on='mes_registro', how='left')

# Recalcular el porcentaje de retención correctamente
retencion_semanal['porcentaje_retencion'] = (retencion_semanal['activo'] / retencion_semanal['total_usuarios_cohorte']) * 100
retencion_semanal

,mes_registro,semana,activo,semana1_valor,porcentaje_retencion,total_usuarios_cohorte
0,2025-01,1,697,697,42.839582,1627
1,2025-01,2,668,697,41.057160,1627
2,2025-01,3,656,697,40.319607,1627
3,2025-01,4,671,697,41.241549,1627
4,2025-02,1,611,611,42.313019,1444
5,2025-02,2,609,611,42.174515,1444
6,2025-02,3,635,611,43.975069,1444
7,2025-02,4,575,611,39.819945,1444
8,2025-03,1,677,677,41.381418,1636
9,2025-03,2,705,677,43.092910,1636


In [64]:
# La retención se mantiene estable (40 – 44%) en las 4 semanas para todas las cohortes, no cae progresivamente como típicamente  lo hace un funnel. 
# La columna ‘activo’ mide actividad independiente por semana, no retención continua acumulada desde el resgistro.


In [65]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT id_usuario, DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_registro
    FROM users
),
tamano_cohorte AS (
    SELECT mes_registro, COUNT(DISTINCT id_usuario) AS total_usuarios_cohorte
    FROM cohortes
    GROUP BY mes_registro
),
actividad_cohorte AS (
    SELECT c.mes_registro, FLOOR(ua.dias_despues_registro / 7) AS semana, SUM(ua.activo) AS usuarios_activos
    FROM user_activity ua
    JOIN cohortes c ON ua.id_usuario = c.id_usuario
    GROUP BY c.mes_registro, FLOOR(ua.dias_despues_registro / 7)
)
SELECT
    a.mes_registro, a.semana, a.usuarios_activos, t.total_usuarios_cohorte,
    (a.usuarios_activos::float / t.total_usuarios_cohorte) * 100 AS porcentaje_retencion
FROM actividad_cohorte a
JOIN tamano_cohorte t ON a.mes_registro = t.mes_registro
ORDER BY a.mes_registro, a.semana;
'''

cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,mes_registro,semana,usuarios_activos,total_usuarios_cohorte,porcentaje_retencion
0,2025-01-01 00:00:00+00:00,1.0,697.0,1627,42.839582
1,2025-01-01 00:00:00+00:00,2.0,668.0,1627,41.057160
2,2025-01-01 00:00:00+00:00,3.0,656.0,1627,40.319607
3,2025-01-01 00:00:00+00:00,4.0,671.0,1627,41.241549
4,2025-02-01 00:00:00+00:00,1.0,611.0,1444,42.313019
5,2025-02-01 00:00:00+00:00,2.0,609.0,1444,42.174515
6,2025-02-01 00:00:00+00:00,3.0,635.0,1444,43.975069
7,2025-02-01 00:00:00+00:00,4.0,575.0,1444,39.819945
8,2025-03-01 00:00:00+00:00,1.0,677.0,1636,41.381418
9,2025-03-01 00:00:00+00:00,2.0,705.0,1636,43.092910


In [66]:
# Paso 4: Retención por cohortes
# Se calculó la retención semanal para cada cohorte (por mes de registro), resuelto tanto en pandas como en SQL, con resultados idénticos entre ambos métodos.
# La retención se mantiene estable (aproximadamente entre 40 y 44%) en las 4 semanas para todas las cohortes, sin la caída progresiva típica de un funnel acumulativo, 
# sugiere que ‘activo’ mide actividad independiente por semana, no retención continua desde el registro.

---

##  Paso 5: Validar si los cambios generan impacto (test estadístico)

 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversión de los usuarios del grupo control (checkout original) es igual a la tasa de conversión de los usuarios del grupo tratamiento (checkout modificado). No hay diferencia entre ambos grupos.
  - **H₁ (Hipótesis alternativa):** La tasa de conversión de los usuarios del grupo tratamiento (checkout modificado) es mayor que la tasa de conversión de los usuarios del grupo control (checkout original). El cambio sí tuvo un impacto positivo.
   
**Test estadístico:** Test z para dos proporciones. Nos permite comparar si la tasa de conversión de dos grupos independientes (control y tratamiento) es estadísticamente diferente, por lo que podemos evaluar la dirección específica de la hipótesis.
**Nivel de significancia alpha:** 0.05

In [67]:
experimento = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
print(experimento.shape)
experimento.info()
experimento.head()

(10000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [68]:
tasa_conversion = experimento.groupby('variante')['convirtio'].mean()
print(tasa_conversion)

variante
control        0.156898
tratamiento    0.162860
Name: convirtio, dtype: float64


In [69]:
conversiones = experimento.groupby('variante')['convirtio'].sum()
print(conversiones)

tamano_grupos = experimento.groupby('variante')['convirtio'].count()
print(tamano_grupos)

variante
control        779
tratamiento    820
Name: convirtio, dtype: int64
variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64


In [70]:
import numpy as np
from scipy.stats import norm

# Datos
conv_control, conv_tratamiento = 779, 820
n_control, n_tratamiento = 4965, 5035

p_control = conv_control / n_control
p_tratamiento = conv_tratamiento / n_tratamiento

p_combinada = (conv_control + conv_tratamiento) / (n_control + n_tratamiento)
error_estandar = np.sqrt(p_combinada * (1 - p_combinada) * (1/n_control + 1/n_tratamiento))

z = (p_tratamiento - p_control) / error_estandar
p_value = 1 - norm.cdf(z)

print("Estadístico z:", z)
print("Valor p:", p_value)

Estadístico z: 0.8132782986429474
Valor p: 0.20802925819560003


Conclusión: Con un valor p = 0.208, mayor que el nivel de significancia alpha = 0.05, no se rechaza H0. No hay evidencia estadística 
suficiente para afirmar que el cambio en la UI del checkout (tratamiento), mejoró la tasa de conversión respecto al checkout original 
(control),la diferencia observada (0.6 puntos porcentuales) es consistente con variación aleatoria de la muestra.

Recomendación: No se recomienda implementar el cambio de forma generalizada con base solamente en este experimento. 
    Se puede tomar en consideración extender la prueba con más usuarios o durante más tiempo para tomar una decisión bien fundamentada.

---

##  Paso 6: Comunicar los resultados (Dashboard en BI)

 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

##  Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [70]:
# https://drive.google.com/drive/folders/1VQ4_d5WOJ1QaBcOeqmdx0vC3u48rvfQl?usp=sharing